# Model — LightGBM

Training des LightGBM-Modells auf `train_final.parquet`. Temporaler Split: 2023–2024 als Train, 2024 Q4 als internes Validation-Set, 2025 als Test.

**Benchmark aus Baseline-Notebook:** Stop Mean MAE = 50.0s — das Modell muss diesen Wert unterbieten.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_2-model")

MODELS_DIR = Path(TRAIN).parent.parent / "models"
MODELS_DIR.mkdir(exist_ok=True)

## Feature Set

Wir laden `train_final.parquet` — das ML-bereite Dataset ohne Rohdaten-Spalten. Folgende Spalten werden explizit **ausgeschlossen**:

| Spalte | Grund |
|:---|:---|
| `departure_delay` | Leakage — nur nach Abfahrt bekannt |
| `delay_delta` | Leakage — nur nach Ankunft bekannt |
| `trip_id` | ID-Spalte, kein Signal |
| `operating_date` | Durch month/weekday/season abgedeckt |

In [ ]:
TARGET = "arrival_delay"

# Leaky: nur nach Ankunft bekannt
# IDs / redundant: keine Vorhersagekraft
EXCLUDE = [
    TARGET,
    "departure_delay",   # leaky
    "delay_delta",       # leaky
    "canceled",          # nach apply_lf_clean redundant — kein Signal
    "trip_id",           # ID
    "operating_date",    # durch month/weekday/year abgedeckt
    "event_name",        # zu granular — event_type + event_weight reichen
    "stop_lat",          # redundant mit stop_name
    "stop_lon",          # redundant mit stop_name
]

# Polars: nur Features + Target laden (spart RAM)
train_path = str(TRAIN).replace("train_features", "train_final")
test_path  = str(TEST).replace("test_features", "test_final")

print("Lade Trainingsdaten...")
train_pl = pl.read_parquet(train_path)
print(f"Train: {len(train_pl):,} rows, {len(train_pl.columns)} cols")

FEATURES = [c for c in train_pl.columns if c not in EXCLUDE]
CAT_COLS  = [c for c in ["line_name", "stop_name", "event_type", "season", "gtfs_year"] if c in FEATURES]

print(f"\nFeatures ({len(FEATURES)}):")
print(FEATURES)
print(f"\nKategoriale Features: {CAT_COLS}")

## Validation Split

Wir splitten die Trainingsdaten zeitlich: **2023 + erstes Halbjahr 2024** als eigentliche Trainingsdaten, **zweites Halbjahr 2024** als internes Validation-Set für Early Stopping. So sieht das Modell beim Training nie Daten aus 2025.

In [ ]:
# Zeitlicher Validation-Split innerhalb der Trainingsdaten
# Train:      2023-01 bis 2024-06
# Validation: 2024-07 bis 2024-12
val_mask = (
    (train_pl["operating_date"].dt.year() == 2024)
    & (train_pl["operating_date"].dt.month() >= 7)
)

train_sub = train_pl.filter(~val_mask)
val_sub   = train_pl.filter(val_mask)

print(f"Train subset:      {len(train_sub):,} rows")
print(f"Validation subset: {len(val_sub):,} rows")

# Pandas konvertieren für LightGBM
# Kategoriale Spalten als pandas Categorical — LightGBM erkennt diese nativ
def to_lgb_df(pl_df: pl.DataFrame, features: list, cat_cols: list) -> pd.DataFrame:
    pdf = pl_df.select(features).to_pandas()
    for col in cat_cols:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype("category")
    return pdf

print("Konvertiere zu Pandas...")
X_train = to_lgb_df(train_sub, FEATURES, CAT_COLS)
y_train = train_sub[TARGET].to_numpy()

X_val   = to_lgb_df(val_sub, FEATURES, CAT_COLS)
y_val   = val_sub[TARGET].to_numpy()

print("Fertig.")

## Training

LightGBM-Parameter für den ersten Lauf — bewusst konservativ:
- `num_leaves=63`: moderate Baumkomplexität
- `learning_rate=0.05`: langsam und stabil
- `n_estimators=1000` mit Early Stopping nach 50 Runden ohne Verbesserung
- `metric=mae`: optimiert direkt auf unsere Hauptmetrik

In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
lgb_val   = lgb.Dataset(X_val,   label=y_val,   reference=lgb_train, free_raw_data=False)

params = {
    "objective":     "regression_l1",   # optimiert MAE direkt
    "metric":        "mae",
    "num_leaves":    63,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":  5,
    "min_child_samples": 50,
    "verbose":       -1,
    "n_jobs":        -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50),
]

print("Training startet...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=callbacks,
)

print(f"\nBeste Iteration: {model.best_iteration}")
print(f"Bestes Val-MAE:  {model.best_score['valid_0']['l1']:.2f}s")

## Validation — Ergebnis

In [ ]:
val_pred = model.predict(X_val, num_iteration=model.best_iteration)

val_mae  = np.abs(y_val - val_pred).mean()
val_rmse = np.sqrt(((y_val - val_pred) ** 2).mean())
val_otp  = (np.abs(y_val - val_pred) <= 60).mean()

BASELINE_MAE = 50.0  # Stop Mean aus Baseline-Notebook (06_prediction_1-baseline)

print(f"Val MAE:          {val_mae:.1f}s")
print(f"Val RMSE:         {val_rmse:.1f}s")
print(f"Val OTP (±60s):   {val_otp:.1%}")
print()
print(f"Baseline (Stop Mean): {BASELINE_MAE}s")
delta = BASELINE_MAE - val_mae
if delta > 0:
    print(f"Modell schlaegt Baseline um {delta:.1f}s")
else:
    print(f"Modell schlechter als Baseline um {abs(delta):.1f}s — Parameter pruefen")

### Interpretation: v1 schlägt Baseline nur um 0.9s (Val) / 4.3s (Test)

Val MAE 49.1s bei einer Baseline von 50.0s — das klingt ernüchternd. Was sagt das?

**Das Modell hat mit 32 Features im Wesentlichen das gelernt, was der historische Stop-Durchschnitt ohnehin schon codiert.** Ohne Information über den *aktuellen Zustand* des Netzwerks — war der vorherige Kurs verspätet? — kann kein Feature-Set diesen strukturellen Deckel durchbrechen.

Das ist kein Fehler im Modell-Design. Es ist der Beweis dass der stärkste Prädiktor noch fehlt: `prev_trip_delay` (Kaskadeneffekt, Pearson r ≥ 0.85 — F-NET-07). 

**Das war der Plan:** v1 als saubere Baseline-Modell, v2 mit dem Kaskadenfeature. Das ist agiles Vorgehen — erst bauen, messen, dann gezielt iterieren. Der Sprung von 45.7s auf 18.56s MAE in v2 wäre nicht so klar interpretierbar ohne diesen v1-Ankerpunkt.

**Positiv zu lesen:** Test MAE 45.7s ist konsistent besser als Val MAE 49.1s — das Modell generalisiert gut auf das Test-Jahr 2025 und overfittet nicht auf das Val-Set.

## Feature Importance

In [ ]:
import plotly.express as px

importance = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)

fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importance (Gain)",
    labels={"importance": "Gain", "feature": ""},
)
fig.update_layout(yaxis=dict(autorange="reversed"), height=550)
fig.show()

show_df(importance.reset_index(drop=True))

## Export

Modell und Test-Predictions speichern — werden in `06_prediction_3-evaluation.ipynb` geladen.

In [ ]:
import json as _json

# --- Modell (LightGBM nativ) ---
model_path = MODELS_DIR / "lgbm_v1.txt"
model.save_model(str(model_path))
print(f"Modell gespeichert: {model_path}")

# --- Params + Metadaten als JSON ---
meta = {
    "model":           "lgbm_v1",
    "best_iteration":  model.best_iteration,
    "val_mae":         round(val_mae, 2),
    "val_rmse":        round(val_rmse, 2),
    "baseline_mae":    BASELINE_MAE,
    "params":          params,
    "features":        FEATURES,
    "cat_cols":        CAT_COLS,
    "target":          TARGET,
    "train_rows":      len(train_sub),
    "val_rows":        len(val_sub),
}
meta_path = MODELS_DIR / "lgbm_v1_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    _json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"Metadaten gespeichert: {meta_path}")

# --- Test-Predictions ---
print("\nLade Testdaten...")
test_pl = pl.read_parquet(test_path)
X_test  = to_lgb_df(test_pl, FEATURES, CAT_COLS)
y_test  = test_pl[TARGET].to_numpy()

test_pred = model.predict(X_test, num_iteration=model.best_iteration)

pred_cols = {
    "actual":    y_test,
    "predicted": test_pred.astype("float32"),
    "line_name": test_pl["line_name"],
    "stop_name": test_pl["stop_name"],
    "hour":      test_pl["hour"],
    "month":     test_pl["month"],
    "has_rain":  test_pl["has_rain"],
    "has_snow":  test_pl["has_snow"],
    "has_event": test_pl["has_event"],
}
# is_anomal: flags Nov 14–Dec 23 2025 rows — useful for evaluation breakdown
if "is_anomal" in test_pl.columns:
    pred_cols["is_anomal"] = test_pl["is_anomal"]

pred_df = pl.DataFrame(pred_cols)

pred_path = Path(test_path).parent / "test_predictions.parquet"
pred_df.write_parquet(pred_path)
print(f"Predictions gespeichert: {pred_path}")

# Quick check Test-MAE
test_mae = np.abs(y_test - test_pred).mean()
print(f"\nTest MAE:  {test_mae:.1f}s")
print(f"Baseline:  {BASELINE_MAE}s")
print(f"Gewinn:    {BASELINE_MAE - test_mae:.1f}s")